# Hari 2 · Sesi 2 — Model, Validasi & Submission
Target sesi: dari data bersih → **skor validasi** → **file submission.csv** yang valid → masuk **leaderboard**.

Alur wajib:
```
split train/validasi → fit → predict → skor (bandingkan baseline) → fit ulang di SEMUA data → predict test → submission.csv
```

**Cara kerja tiap latihan**
1. Baca **Contoh**, lalu jalankan: klik selnya, tekan `Shift + Enter`.
2. Kerjakan **Latihan**: ganti `____` (atau `...`) dengan kode yang benar, lalu jalankan. Baris `cek(...)` di bawahnya langsung bilang ✅ atau ❌.
3. Mentok? Jalankan `hint('1.2')` di sel baru (bisa dipanggil berkali-kali, petunjuknya makin jelas). Masih mentok, baru `jawaban('1.2')`. Setelah lihat jawaban, **ketik ulang sendiri, jangan copy-paste**.
4. Kalau muncul error merah, baca **baris terakhirnya** dulu. Di bawahnya ada **💡 PENJELASAN DOJO** dalam bahasa Indonesia.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from cek import cek, hint, jawaban, progres, skor

# ===== Preprocessing kilat (hasil sesi 03, dirangkum) =====
train = pd.read_csv('data/titanic/train.csv')
test = pd.read_csv('data/titanic/test.csv')
y = train['Survived']
X = train.drop(columns=['Survived'])
X_test = test.copy()
test_ids = test['PassengerId']

for df in [X, X_test]:                      # lakukan hal yang sama ke train & test
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['HasCabin'] = df['Cabin'].notnull().astype(int)

X = X.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
X_test = X_test.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

for kol in ['Age', 'Fare']:
    median = X[kol].median()
    X[kol] = X[kol].fillna(median)
    X_test[kol] = X_test[kol].fillna(median)

modus = X['Embarked'].mode()[0]
X['Embarked'] = X['Embarked'].fillna(modus)
X_test['Embarked'] = X_test['Embarked'].fillna(modus)

X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})
X_test['Sex'] = X_test['Sex'].map({'male': 0, 'female': 1})

X = pd.get_dummies(X, columns=['Embarked'], dtype=int)
X_test = pd.get_dummies(X_test, columns=['Embarked'], dtype=int)
X_test = X_test.reindex(columns=X.columns, fill_value=0)
print('Siap! X:', X.shape, '| X_test:', X_test.shape)

## 1. Bagi data: train & validasi
Kita **tidak tahu** jawaban test. Jadi sebagian train (20%) disisihkan sebagai **ujian latihan** (validasi) untuk mengukur model sebelum submit.
- `test_size=0.2` → 20% untuk validasi
- `random_state=42` → pembagiannya sama setiap kali dijalankan
- `stratify=y` (klasifikasi saja) → proporsi kelas di train & validasi sama

**Latihan 4.1** — Bagi `X`, `y` jadi 80% train, 20% validasi, dengan stratify.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=____, random_state=42, stratify=____)
cek('4.1', X_train, X_val, y_train, y_val)

**Latihan 4.2** — Hitung akurasi **baseline** di data validasi: tebak kelas terbanyak untuk semua orang.

In [ ]:
tebakan_mayoritas = y_train.mode()[0]
baseline_acc = (y_val == ____).mean()
cek('4.2', baseline_acc)
print('Baseline:', baseline_acc)

## 2. Model pertama: Random Forest
Pola 3 langkah yang **selalu sama** untuk semua model scikit-learn:
```python
model = NamaModel()             # 1. buat
model.fit(X_train, y_train)     # 2. latih
pred = model.predict(X_val)     # 3. tebak
```

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model_rf = RandomForestClassifier(n_estimators=200, random_state=42)
model_rf.fit(X_train, y_train)
pred_val = model_rf.predict(X_val)
print('Akurasi Random Forest:', accuracy_score(y_val, pred_val))

**Latihan 4.3** — Coba **Logistic Regression** dengan pola yang sama.

In [ ]:
from sklearn.linear_model import LogisticRegression
model_lr = LogisticRegression(max_iter=1000)
model_lr.____(X_train, y_train)
pred_lr = model_lr.____(X_val)
acc_lr = accuracy_score(____, ____)
cek('4.3', acc_lr)
print('Akurasi LR:', acc_lr)

**Latihan 4.4** — Dari ingatan: coba **Decision Tree** (`DecisionTreeClassifier(random_state=42)`). Simpan model di `model_dt`, akurasinya di `acc_dt`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model_dt = ...
...
acc_dt = ...
cek('4.4', acc_dt)
print('Akurasi DT:', acc_dt)

## 3. Metrik klasifikasi selain akurasi
| Metrik | Kapan dipakai |
|---|---|
| **Accuracy** | kelas seimbang |
| **F1-score** | kelas tidak seimbang, dan salah tebak di kelas minoritas itu penting |
| **ROC-AUC** | kompetisi minta **probabilitas**. Submit `model.predict_proba(X_test)[:, 1]`, bukan 0/1 |

**Wajib:** baca metrik kompetisi di halaman Overview/Evaluation, lalu pakai metrik yang **sama** untuk validasi.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_val, pred_val))       # [[benar-0, salah-1], [salah-0, benar-1]]
print(classification_report(y_val, pred_val))

**Latihan 4.5** — Hitung **F1-score** Random Forest di validasi (pakai `y_val` dan `pred_val`).

In [ ]:
from sklearn.metrics import f1_score
f1_rf = f1_score(____, ____)
cek('4.5', f1_rf)
print('F1:', f1_rf)

## 4. Model final → prediksi test → submission
Setelah memilih model terbaik dari validasi, **latih ulang pakai SEMUA data train** (`X`, `y`) supaya model belajar dari data sebanyak mungkin. Baru setelah itu predict `X_test`.

**Latihan 4.6** — Latih `model_final` di **semua** data, lalu prediksi `X_test`.

In [ ]:
model_final = RandomForestClassifier(n_estimators=200, random_state=42)
model_final.fit(____, ____)
pred_test = model_final.predict(____)
cek('4.6', pred_test)

**Latihan 4.7** — Buat `submission` berisi kolom `PassengerId` (dari `test_ids`) dan `Survived` (dari `pred_test`), lalu simpan ke `submission.csv` **tanpa index**.

In [ ]:
submission = pd.DataFrame({'PassengerId': ____, 'Survived': ____})
submission.to_csv('submission.csv', index=____)
cek('4.7', submission)
submission.head()

In [ ]:
# Leaderboard lokal: skor file submission-mu terhadap kunci jawaban test
skor('titanic', 'submission.csv')

## 5. ⭐ Jurus pamungkas: Pipeline (anti-lupa, anti-error)
Langkah 3–6 di notebook 03 bisa dirangkum dalam **satu pipeline**. Keuntungannya:
- Otomatis mengisi kosong, men-scale, dan one-hot.
- Otomatis memakai ukuran dari **train** untuk test, jadi tidak ada kebocoran.
- Tidak ada masalah kolom train ≠ test. Kategori baru di test pun aman (`handle_unknown='ignore'`).

```
data mentah ─┬─ kolom angka ─→ isi kosong (median) → scaling ─────┬─→ model
             └─ kolom teks  ─→ isi kosong (paling sering) → one-hot ┘
```
Pilihan `strategy` untuk SimpleImputer: `'median'`, `'mean'`, `'most_frequent'`, `'constant'`.

**Ini yang dipakai di TEMPLATE_KOMPETISI.** Hafalkan bentuknya.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Mulai dari data MENTAH (tanpa preprocessing manual)
train_raw = pd.read_csv('data/titanic/train.csv')
test_raw = pd.read_csv('data/titanic/test.csv')
BUANG = ['PassengerId', 'Name', 'Ticket', 'Cabin']
X_raw = train_raw.drop(columns=['Survived'] + BUANG)
y_raw = train_raw['Survived']
X_test_raw = test_raw.drop(columns=BUANG)
num_cols = X_raw.select_dtypes(include='number').columns.tolist()
cat_cols = X_raw.select_dtypes(exclude='number').columns.tolist()
print('Angka:', num_cols, '| Teks:', cat_cols)

**Latihan 4.8** — Lengkapi pipeline: angka diisi **median**, teks diisi **paling sering** (`'most_frequent'`), dan OneHotEncoder **mengabaikan** kategori baru.

In [ ]:
preprocess = ColumnTransformer([
    ('num', Pipeline([('isi_kosong', SimpleImputer(strategy=____)),
                      ('skala', StandardScaler())]), num_cols),
    ('kat', Pipeline([('isi_kosong', SimpleImputer(strategy=____)),
                      ('onehot', OneHotEncoder(handle_unknown=____))]), cat_cols),
])
pipe = Pipeline([('prep', preprocess),
                 ('model', RandomForestClassifier(n_estimators=200, random_state=42))])
pipe.fit(X_raw, y_raw)
cek('4.8', pipe)

In [ ]:
# Pipeline dipakai persis seperti model biasa
pred_pipe = pipe.predict(X_test_raw)
pd.DataFrame({'PassengerId': test_raw['PassengerId'], 'Survived': pred_pipe}).to_csv('submission_pipeline.csv', index=False)
skor('titanic', 'submission_pipeline.csv')

## 6. Kalau kompetisinya REGRESI (menebak angka)
Bedanya cuma dua: **model** (`...Regressor`) dan **metrik**.
| Metrik | Arti | Bagus kalau |
|---|---|---|
| **RMSE** | akar rata-rata kuadrat error, satuannya sama dengan target, menghukum error besar | makin **kecil** |
| **MAE** | rata-rata selisih absolut | makin **kecil** |
| **R²** | seberapa banyak variasi yang dijelaskan (maks 1) | makin **dekat 1** |

Contoh: menebak `Fare` dari kolom lain.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Xf = X.drop(columns=['Fare'])      # fitur
yf = X['Fare']                     # target regresi
Xf_train, Xf_val, yf_train, yf_val = train_test_split(Xf, yf, test_size=0.2, random_state=42)   # regresi: TANPA stratify
model_reg = RandomForestRegressor(n_estimators=200, random_state=42)
model_reg.fit(Xf_train, yf_train)
pred_fare = model_reg.predict(Xf_val)
print('R2:', r2_score(yf_val, pred_fare))

**Latihan 4.9** — Hitung **RMSE** dan **MAE** dari `yf_val` dan `pred_fare`.

In [ ]:
rmse = np.sqrt(mean_squared_error(____, ____))
mae = mean_absolute_error(____, ____)
cek('4.9', rmse, mae)
print('RMSE:', rmse, '| MAE:', mae)

## 7. Bonus: cross-validation (skor validasi yang lebih bisa dipercaya)
Satu kali split bisa "beruntung" atau "sial". Cross-validation membagi data jadi 5 bagian, lalu bergiliran jadi validasi, dan hasilnya dirata-rata.

**Latihan 4.10** — Hitung rata-rata akurasi 5-fold cross-validation Random Forest di `X`, `y`.

In [ ]:
from sklearn.model_selection import cross_val_score
skor_cv = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), X, y, cv=____, scoring='accuracy')
rata_cv = skor_cv.____()
cek('4.10', rata_cv)
print(skor_cv.round(3), '→ rata-rata', round(rata_cv, 4))

In [ ]:
progres()

**Selesai!** Kamu sudah melewati satu kompetisi utuh. Berikutnya: **05_MOCK1_Churn.ipynb**, simulasi kompetisi dengan timer, pakai **TEMPLATE_KOMPETISI.ipynb** dan contekan.